# YOLOv11m Point Detection 성능 평가
베스트 모델을 로드하여 논문용 종합 성능 평가를 수행합니다.

In [ ]:
# 필요한 라이브러리 import
import copy
import csv
import os
import warnings
from argparse import ArgumentParser
import numpy as np
import torch
from tqdm import tqdm
import yaml
from torch.utils import data
import glob
import json
from nets import nn
from utils import util
import pandas as pd
from utils.dataset import Dataset
from torch.utils import data
import numpy as np
import cv2
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
from utils.valid import compute_point_label_metrics_single
from utils.valid import visualize_ground_truth_and_prediction_separately_detail_single

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print("device:", device)

with open('utils/detail_args.yaml', errors='ignore') as f:
    params = yaml.safe_load(f)

In [ ]:
# collate 함수 정의
def collate_fn1(batch):
    """배치 collation (단일 이미지만)"""
    samples, cls, box, indices = zip(*batch)

    cls = torch.cat(cls, dim=0)
    box = torch.cat(box, dim=0)

    new_indices = list(indices)
    for i in range(len(indices)):
        new_indices[i] += i
    indices = torch.cat(new_indices, dim=0)

    targets = {'cls': cls,
                'box': box,
                'idx': indices}
    return torch.stack(samples, dim=0), targets

# 데이터셋 클래스 정의
class custom_dataset(data.Dataset):
    def __init__(self, images, params, augment, labels, image_infos=None):
        self.params = params
        self.mosaic = augment
        self.augment = augment
        self.images = images
        self.labels = labels
        self.scale_list = [256, 288, 320, 352, 384, 416, 448, 480, 512]
        self.current_scale = 512
        self.n = len(self.images)
        self.indices = list(range(self.n))
    
    def __len__(self):
        return len(self.indices)
    
    def set_scale(self, scale):
        """배치 단위로 스케일 설정"""
        self.current_scale = scale
    
    def random_multi_scale(self, image, label):
        size = self.current_scale
        image = cv2.resize(image, (size, size))
        temp_labels = []
        for i in range(len(label)):
            x_center = label[i][1]
            y_center = label[i][2]
            w = label[i][3]
            h = label[i][4]
            temp_labels.append([label[i][0], y_center, x_center, h, w])
        return image, temp_labels
    
    def __getitem__(self, index):
        index = self.indices[index]
        image = self.images[index].copy()
        label = self.labels[index]
        
        image, label = self.random_multi_scale(image, label)
        cls = []
        box = []
        for i in range(len(label)):
            cls.append(label[i][0])
            box.append(label[i][1:5])
        cls = np.array(cls)
        box = np.array(box)
        nl = len(box)
        
        image = image.transpose((2, 0, 1))
        
        return (torch.from_numpy(image), 
                torch.from_numpy(cls), 
                torch.from_numpy(box), 
                torch.zeros(nl))

In [ ]:
# 데이터 로딩
print("데이터 로딩 시작...")
input_size = 512
label_dir = '../../data/spatialTranscriptome/detail_preprocessed_xenium/patch_train_data/**/annotation/'
label_files = sorted(glob.glob(f"{label_dir}/*.csv"))
    
image_filenames = []
labels = []

for i in tqdm(range(len(label_files)), desc="라벨 파일 로딩"):
    label_file = label_files[i]
    data1 = pd.read_csv(label_file)
    img_path = label_file.replace('/annotation', '/image').replace('.csv', '.png')
    if os.path.exists(img_path):
        image_filenames.append(img_path)
        arr = data1.to_numpy()
        arr = arr[:, [4, 1, 0, 3, 2]]  # class, y, x, h, w
        temp_labels = list(arr)
        labels.append(temp_labels)

images = []
for i in tqdm(range(len(image_filenames)), desc="이미지 로딩"):
    image = cv2.imread(image_filenames[i])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    if image.shape[0] != 512 or image.shape[1] != 512:
        image = cv2.copyMakeBorder(image, 0, 512 - image.shape[0], 0, 512 - image.shape[1], cv2.BORDER_CONSTANT, value=[255, 255, 255])
    images.append(image)

print(f"총 이미지 수: {len(images)}")
print(f"총 라벨 수: {len(labels)}")

In [ ]:
# 데이터셋 분할 및 생성
patch_train, patch_test, label_train, label_test = train_test_split(
    images, labels, 
    test_size=0.1,
    random_state=242,
    shuffle=True
)

val_dataset = custom_dataset(patch_test, params, augment=False, labels=label_test)
val_dataset.set_scale(512)

batch_size = 16
val_loader = torch.utils.data.DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn1,
    drop_last=False
)

print(f"검증 데이터셋 크기: {len(val_dataset)}")
print(f"검증 배치 수: {len(val_loader)}")

In [ ]:
# 모델 로딩
save_dir = '../../model/spatialTranscriptome/detail_yolov11m_patch_train/'
model = nn.yolo_v11_m(len(params['names'])).to(device)

print("베스트 모델 로딩...")
best_checkpoint = torch.load(os.path.join(save_dir, 'best_model.pt'), map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

print(f"\n✅ 베스트 모델 로드 완료 (Epoch {best_checkpoint['epoch']})")
print(f"   - Detection Recall: {best_checkpoint['detection_recall']:.4f}")
print(f"   - Classification Accuracy: {best_checkpoint['classification_accuracy']:.4f}")

In [ ]:
# =======================================================================================
# 논문용 종합 성능 평가 (Publication-Ready Performance Evaluation)
# Point Detection 중심 평가: Detection Recall + Classification Accuracy
# =======================================================================================

print("=" * 90)
print("📊 논문용 종합 성능 평가 시작 (Point Detection Focused)")
print("=" * 90)

# =======================================================================================
# 1. Distance Threshold 민감도 분석
# =======================================================================================
print("\n" + "=" * 90)
print("1️⃣ Distance Threshold 민감도 분석")
print("=" * 90)

distance_thresholds = [4, 8, 12, 16, 20, 24, 32]
threshold_results = {
    'threshold': [],
    'detection_recall': [],
    'classification_accuracy': []
}

for thresh in distance_thresholds:
    print(f"  - Threshold {thresh}px 평가 중...", end=" ")
    metrics = compute_point_label_metrics_single(
        model, val_loader, device, params, distance_threshold=thresh
    )
    threshold_results['threshold'].append(thresh)
    threshold_results['detection_recall'].append(metrics['detection_recall'])
    threshold_results['classification_accuracy'].append(metrics['classification_accuracy'])
    
    print(f"Detection={metrics['detection_recall']:.4f}, Classification={metrics['classification_accuracy']:.4f}")

# =======================================================================================
# 2. 클래스별 상세 Confusion Matrix 생성 (Distance Threshold: 16px)
# =======================================================================================
print("\n" + "=" * 90)
print("2️⃣ Confusion Matrix 및 Classification Report 생성 (Distance Threshold: 16px)")
print("=" * 90)

def compute_distance_matrix_np(gt_centers, pred_centers):
    """거리 행렬 계산"""
    if len(gt_centers) == 0 or len(pred_centers) == 0:
        return np.array([])
    
    gt_centers = np.array(gt_centers)
    pred_centers = np.array(pred_centers)
    
    distances = np.sqrt(
        ((gt_centers[:, None, 0] - pred_centers[None, :, 0]) ** 2) +
        ((gt_centers[:, None, 1] - pred_centers[None, :, 1]) ** 2)
    )
    return distances

all_gt_labels = []
all_pred_labels = []
all_confidences = []

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Confusion Matrix 계산"):
        images = images.to(device).float() / 255.0
        
        with torch.amp.autocast('cuda'):
            pred = model(images)
        
        results = util.non_max_suppression(pred, confidence_threshold=0.15, iou_threshold=0.45)
        
        for i in range(len(images)):
            # GT 추출
            cls_targets = targets['cls']
            box_targets = targets['box']
            idx_targets = targets['idx']
            
            batch_mask = idx_targets == i
            if not batch_mask.any():
                continue
            
            batch_cls = cls_targets[batch_mask].cpu().numpy()
            batch_box = box_targets[batch_mask].cpu().numpy()
            
            # GT 중심점 (픽셀 좌표)
            img_size = 512
            gt_centers = []
            for box in batch_box:
                x_center = box[0] * img_size
                y_center = box[1] * img_size
                gt_centers.append([x_center, y_center])
            gt_centers = np.array(gt_centers)
            
            if len(gt_centers) == 0:
                continue
            
            # 예측 처리
            if len(results) > i and len(results[i]) > 0:
                pred_boxes = results[i][:, :4].cpu().numpy()
                pred_classes = results[i][:, 5].cpu().numpy()
                pred_scores = results[i][:, 4].cpu().numpy()
                
                # 예측 중심점
                pred_centers = []
                for box in pred_boxes:
                    x_center = (box[0] + box[2]) / 2
                    y_center = (box[1] + box[3]) / 2
                    pred_centers.append([x_center, y_center])
                pred_centers = np.array(pred_centers)
                
                # 거리 행렬 및 Hungarian matching
                distance_matrix = compute_distance_matrix_np(gt_centers, pred_centers)
                
                if distance_matrix.size > 0:
                    gt_indices, pred_indices = linear_sum_assignment(distance_matrix)
                    
                    matched_gt = set()
                    for gt_idx, pred_idx in zip(gt_indices, pred_indices):
                        if distance_matrix[gt_idx, pred_idx] <= 16:
                            matched_gt.add(gt_idx)
                            all_gt_labels.append(int(batch_cls[gt_idx]))
                            all_pred_labels.append(int(pred_classes[pred_idx]))
                            all_confidences.append(float(pred_scores[pred_idx]))
                    
                    # 매칭 실패한 GT (False Negative)
                    for gt_idx in range(len(gt_centers)):
                        if gt_idx not in matched_gt:
                            all_gt_labels.append(int(batch_cls[gt_idx]))
                            all_pred_labels.append(-1)  # -1: missed detection
            else:
                # 예측 없음
                for cls_id in batch_cls:
                    all_gt_labels.append(int(cls_id))
                    all_pred_labels.append(-1)

# Confusion Matrix 계산 (매칭된 것만)
matched_indices = [i for i, pred in enumerate(all_pred_labels) if pred != -1]
if len(matched_indices) > 0:
    matched_gt = [all_gt_labels[i] for i in matched_indices]
    matched_pred = [all_pred_labels[i] for i in matched_indices]
    
    num_classes = len(params['names'])
    cm = confusion_matrix(matched_gt, matched_pred, labels=list(range(num_classes)))
    
    detection_recall = len(matched_gt) / len(all_gt_labels)
    classification_accuracy = sum(1 for g, p in zip(matched_gt, matched_pred) if g == p) / len(matched_gt)
    
    print(f"\n총 GT 포인트: {len(all_gt_labels)}")
    print(f"매칭된 포인트: {len(matched_gt)} ({detection_recall*100:.2f}%) - Detection Recall")
    print(f"놓친 포인트 (FN): {all_pred_labels.count(-1)} ({all_pred_labels.count(-1)/len(all_gt_labels)*100:.2f}%)")
    print(f"분류 정확도: {classification_accuracy:.4f} - Classification Accuracy")
    
    # Classification Report
    class_names_list = [f"Class {i}" for i in range(num_classes)]
    print("\n📊 Classification Report (매칭된 객체 대상):")
    print(classification_report(matched_gt, matched_pred, target_names=class_names_list, digits=4))

In [ ]:
# =======================================================================================
# 3. 시각화 생성 (6-panel figure) - Point Detection 중심
# =======================================================================================
print("\n" + "=" * 90)
print("3️⃣ 논문용 종합 시각화 생성 (Point Detection Focused)")
print("=" * 90)

# 학습 히스토리가 없으므로 생략하고 주요 그래프만 생성
fig = plt.figure(figsize=(18, 6))

# 3.1 Distance Threshold Analysis
ax1 = plt.subplot(1, 3, 1)
ax1.plot(threshold_results['threshold'], threshold_results['detection_recall'], 
         'o-', label='Detection Recall', linewidth=3, markersize=12, color='#2E86AB')
ax1.plot(threshold_results['threshold'], threshold_results['classification_accuracy'], 
         's-', label='Classification Acc', linewidth=3, markersize=12, color='#F18F01')
ax1.set_xlabel('Distance Threshold (pixels)', fontsize=16, fontweight='bold')
ax1.set_ylabel('Score', fontsize=16, fontweight='bold')
ax1.set_title('Performance vs Distance Threshold', fontsize=18, fontweight='bold', pad=20)
ax1.legend(fontsize=14, frameon=True, shadow=True, loc='lower right')
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_ylim([0, 1.05])
ax1.tick_params(labelsize=12)

# 3.2 Confusion Matrix (Normalized)
ax2 = plt.subplot(1, 3, 2)
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)
cm_normalized = np.nan_to_num(cm_normalized)
sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Blues', 
            xticklabels=[f'C{i}' for i in range(num_classes)],
            yticklabels=[f'C{i}' for i in range(num_classes)],
            ax=ax2, cbar_kws={'label': 'Proportion'}, vmin=0, vmax=1,
            annot_kws={'size': 14, 'weight': 'bold'})
ax2.set_xlabel('Predicted Class', fontsize=16, fontweight='bold')
ax2.set_ylabel('True Class', fontsize=16, fontweight='bold')
ax2.set_title('Normalized Confusion Matrix', fontsize=18, fontweight='bold', pad=20)
ax2.tick_params(labelsize=12)

# 3.3 Confusion Matrix (Raw Counts)
ax3 = plt.subplot(1, 3, 3)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=[f'C{i}' for i in range(num_classes)],
            yticklabels=[f'C{i}' for i in range(num_classes)],
            ax=ax3, cbar_kws={'label': 'Count'},
            annot_kws={'size': 14, 'weight': 'bold'})
ax3.set_xlabel('Predicted Class', fontsize=16, fontweight='bold')
ax3.set_ylabel('True Class', fontsize=16, fontweight='bold')
ax3.set_title('Confusion Matrix (Counts)', fontsize=18, fontweight='bold', pad=20)
ax3.tick_params(labelsize=12)

plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'test_evaluation.png'), 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(os.path.join(save_dir, 'test_evaluation.pdf'), 
            bbox_inches='tight', facecolor='white')
print(f"✅ 평가 그래프 저장: {save_dir}/test_evaluation.png/pdf")
plt.show()

In [ ]:
# =======================================================================================
# 4. 성능 테이블 생성 및 저장
# =======================================================================================
print("\n" + "=" * 90)
print("4️⃣ 성능 테이블 생성 및 CSV 저장")
print("=" * 90)

# Overall Performance Table
performance_data = {
    'Metric': ['Detection Recall', 'Classification Accuracy'],
    'Score': [
        f"{detection_recall:.4f}",
        f"{classification_accuracy:.4f}"
    ],
    'Description': [
        'GT 중심점 중 찾은 비율 (세포 발견율)',
        '찾은 세포의 클래스 분류 정확도'
    ]
}
performance_df = pd.DataFrame(performance_data)

print("\n📊 핵심 성능 지표 (Point Detection):")
print(performance_df.to_string(index=False))

# Per-Class Performance Table
class_data = {
    'Class': [],
    'Total_GT': [],
    'Detected': [],
    'Detection_Rate': [],
    'Correctly_Classified': [],
    'Classification_Acc': []
}

num_classes = len(params['names'])
for cid in range(num_classes):
    total_gt = all_gt_labels.count(cid)
    detected_count = sum(1 for gt, pred in zip(all_gt_labels, all_pred_labels) 
                        if gt == cid and pred != -1)
    correctly_classified = sum(1 for gt, pred in zip(all_gt_labels, all_pred_labels) 
                               if gt == cid and pred == cid)
    
    det_rate = detected_count / total_gt if total_gt > 0 else 0
    cls_acc = correctly_classified / detected_count if detected_count > 0 else 0
    
    class_data['Class'].append(f'Class {cid}')
    class_data['Total_GT'].append(total_gt)
    class_data['Detected'].append(detected_count)
    class_data['Detection_Rate'].append(f"{det_rate:.4f}")
    class_data['Correctly_Classified'].append(correctly_classified)
    class_data['Classification_Acc'].append(f"{cls_acc:.4f}")

class_df = pd.DataFrame(class_data)
print("\n📊 클래스별 성능 (Point Detection - Confusion Matrix 기반):")
print(class_df.to_string(index=False))

# Distance Threshold Analysis Table
threshold_df = pd.DataFrame(threshold_results)
threshold_df = threshold_df.round(4)
print("\n📊 Distance Threshold 민감도 분석:")
print(threshold_df.to_string(index=False))

# CSV 저장
performance_df.to_csv(os.path.join(save_dir, 'test_overall.csv'), index=False)
class_df.to_csv(os.path.join(save_dir, 'test_per_class.csv'), index=False)
threshold_df.to_csv(os.path.join(save_dir, 'test_threshold_analysis.csv'), index=False)

print(f"\n✅ 성능 테이블 CSV 저장 완료: {save_dir}")

In [ ]:
# =======================================================================================
# 5. 대표 샘플 시각화 (Best Cases)
# =======================================================================================
print("\n" + "=" * 90)
print("5️⃣ 대표 샘플 예측 시각화 (Best Cases)")
print("=" * 90)

sample_scores = []

with torch.no_grad():
    for idx in tqdm(range(len(val_dataset)), desc="샘플별 점수 계산"):
        image, cls_tensor, box_tensor, _ = val_dataset[idx]
        image = image.unsqueeze(0).to(device).float() / 255.0
        
        batch_cls = cls_tensor.numpy()
        batch_box = box_tensor.numpy()
        
        img_size = 512
        gt_centers = []
        for box in batch_box:
            x_center = box[0] * img_size
            y_center = box[1] * img_size
            gt_centers.append([x_center, y_center])
        gt_centers = np.array(gt_centers)
        
        if len(gt_centers) == 0:
            continue
        
        with torch.amp.autocast('cuda'):
            pred = model(image)
        results = util.non_max_suppression(pred, confidence_threshold=0.15, iou_threshold=0.45)
        
        if len(results[0]) > 0:
            pred_boxes = results[0][:, :4].cpu().numpy()
            pred_classes = results[0][:, 5].cpu().numpy()
            
            pred_centers = []
            for box in pred_boxes:
                x_center = (box[0] + box[2]) / 2
                y_center = (box[1] + box[3]) / 2
                pred_centers.append([x_center, y_center])
            pred_centers = np.array(pred_centers)
            
            distance_matrix = compute_distance_matrix_np(gt_centers, pred_centers)
            
            if distance_matrix.size > 0:
                gt_indices, pred_indices = linear_sum_assignment(distance_matrix)
                
                detected = 0
                correct = 0
                for gt_idx, pred_idx in zip(gt_indices, pred_indices):
                    if distance_matrix[gt_idx, pred_idx] <= 16:
                        detected += 1
                        if batch_cls[gt_idx] == pred_classes[pred_idx]:
                            correct += 1
                
                detection_rate = detected / len(gt_centers)
                classification_rate = correct / detected if detected > 0 else 0
                combined_score = (detection_rate + classification_rate) / 2
            else:
                combined_score = 0
        else:
            combined_score = 0
        
        sample_scores.append((idx, combined_score))

# 상위 3개 선택
sample_scores.sort(key=lambda x: x[1], reverse=True)
best_samples = sample_scores[:3]

print(f"\n✨ Best samples (Combined Score): {[(idx, f'{score:.3f}') for idx, score in best_samples]}")

# Best samples 시각화
for rank, (idx, score) in enumerate(best_samples, 1):
    print(f"\n  Best Sample #{rank} (idx={idx}, Score={score:.3f}) 시각화 중...")
    visualize_ground_truth_and_prediction_separately_detail_single(
        model, val_dataset, idx=idx, epoch=f'test_best_{rank}', 
        save_dir=save_dir
    )

print("\n" + "=" * 90)
print("✅ 테스트 평가 완료!")
print("=" * 90)
print(f"\n📁 저장된 파일:")
print(f"  - test_evaluation.png/pdf : 3-panel 분석")
print(f"  - test_overall.csv : 전체 성능 지표")
print(f"  - test_per_class.csv : 클래스별 성능 (Distance: 16px)")
print(f"  - test_threshold_analysis.csv : Distance threshold 분석")
print(f"  - test_best_*.png : 최상위 예측 샘플 (Top 3)")
print(f"\n모든 파일 위치: {save_dir}")
print(f"\n🎯 테스트 성능:")
print(f"  - Detection Recall: {detection_recall:.4f} ⭐ (세포 발견율)")
print(f"  - Classification Accuracy: {classification_accuracy:.4f} ⭐ (분류 정확도)")
print(f"  - Distance Threshold: 16px (클래스별 성능 및 Confusion Matrix)")